# Silver Layer — Clean & Conform Education Data
Dedupe, cast types. Keeps enrolment label is_withdrawn; FE drops post-outcome leakage.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_date, row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Clean students
df_s = spark.read.format('delta').table('bronze_students')
w = Window.partitionBy('student_id').orderBy(col('ingestion_timestamp').desc())
df_s = (
    df_s.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('cohort_year', col('cohort_year').cast('int'))
    .withColumn('age_at_enrolment', col('age_at_enrolment').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_s.write.mode('overwrite').format('delta').saveAsTable('silver_students')
print(f'silver_students: {spark.table("silver_students").count()} rows')

In [ ]:
# Clean faculty
df_f = spark.read.format('delta').table('bronze_faculty')
w2 = Window.partitionBy('faculty_id').orderBy(col('ingestion_timestamp').desc())
df_f = (
    df_f.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('years_at_institution', col('years_at_institution').cast('int'))
    .withColumn('courses_assigned', col('courses_assigned').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_f.write.mode('overwrite').format('delta').saveAsTable('silver_faculty')
print(f'silver_faculty: {spark.table("silver_faculty").count()} rows')

In [ ]:
# Clean enrolments (keep label is_withdrawn)
df_e = spark.read.format('delta').table('bronze_enrolments')
w3 = Window.partitionBy('enrolment_id').orderBy(col('ingestion_timestamp').desc())
df_e = (
    df_e.withColumn('_rn', row_number().over(w3)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('credits', col('credits').cast('int'))
    .withColumn('age_at_enrolment', col('age_at_enrolment').cast('int'))
    .withColumn('is_completed', col('is_completed').cast('int'))
    .withColumn('is_withdrawn', col('is_withdrawn').cast('int'))
    .withColumn('enrolment_date', to_date('enrolment_date'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_e.write.mode('overwrite').format('delta').saveAsTable('silver_enrolments')
print(f'silver_enrolments: {spark.table("silver_enrolments").count()} rows')

In [ ]:
# Clean assessments
df_a = spark.read.format('delta').table('bronze_assessments')
df_a = (
    df_a
    .withColumn('submitted_date', to_date('submitted_date'))
    .withColumn('score', col('score').cast('double'))
    .withColumn('is_pass', col('is_pass').cast('int'))
    .withColumn('attempt_number', col('attempt_number').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_a.write.mode('overwrite').format('delta').saveAsTable('silver_assessments')
print(f'silver_assessments: {spark.table("silver_assessments").count()} rows')